# Lexical Semantics

In [1]:
import numpy as np
import termtables as tt
import nltk

from scipy.stats import pearsonr

nltk.download('wordnet_ic')

from nltk.corpus import wordnet as wn
from nltk.corpus import wordnet_ic

[nltk_data] Downloading package wordnet_ic to
[nltk_data]     /Users/josepdecidrodriguez/nltk_data...
[nltk_data]   Package wordnet_ic is already up-to-date!


In [2]:
words = [
    ('the','DT'), ('man','NN'), ('swim','VB'), ('with', 'PR'), ('a', 'DT'),
    ('girl','NN'), ('and', 'CC'), ('a', 'DT'), ('boy', 'NN'), ('whilst', 'PR'),
    ('the', 'DT'), ('woman', 'NN'), ('walk', 'VB')
]

pos_translation = {
    'NN': 'n',
    'VB': 'v',
    'JJ': 'a',
    'JJR': 'a',
    'JJS': 'a',
    'RB': 'r',
    'RBR': 'r',
    'RBS': 'r'   
}

brown_ic = wordnet_ic.ic('ic-brown.dat')

In [3]:
def compute_similarities(w1, w2):
    pos1 = pos_translation.get(w1[1])
    pos2 = pos_translation.get(w2[1])
    
    # Work only with names, verbs, adjectives or adverbs
    if pos1 is not None and pos2 is not None:
        word1 = wn.synset(f'{w1[0]}.{pos1}.01')
        word2 = wn.synset(f'{w2[0]}.{pos2}.01')
        
        # Returns None if different POS
        path_sim = word1.path_similarity(word2)
        
        # Error if different POS, must check manually
        lch_sim = word1.lch_similarity(word2) if pos1 == pos2 else None
        
        # Returns None if different POS
        wup_sim = word1.wup_similarity(word2)
        
        # Error if different POS, must check manually
        lin_sim = word1.lin_similarity(word2, brown_ic) if pos1 == pos2 else None
        
        return path_sim, lch_sim, wup_sim, lin_sim

In order to compare the similarity methods, we first need to have them all in the same scale. Path, Wu-Palmer and Lin similarities are already in range [0, 1], so we will just scale Leacock-Chodrow similarity to the same range. 

If we imagine the structure as a tree, its minimum value is when both elements are at maximum distance, go up a whole branch from the bottom (max depth of 20) and go down again in another one. Applying the formula it results 0. For the maximum value we don't even need the formula, just compare one specific word with itself, being the distance the minimum. Hence, we will scale the similarities between 0 and this value.

In [4]:
sims = []
results = []

for i, w1 in enumerate(words):
    for j, w2 in enumerate(words):
        if i != j:
            sim = compute_similarities(w1, w2)
            if sim is not None:
                sims.append(list(sim))
                results.append(([w1[0], w2[0]]))
        
# Maximum value for lch similarity is two compare two equal words
lch_word1 = wn.synset('walk.n.01')
lch_word2 = wn.synset('walk.n.01')
lch_max = lch_word1.lch_similarity(lch_word2)

for i in range(len(sims)):
    # We scale these values between 0 and 1
    if sims[i][1] is not None:
        sims[i][1] = sims[i][1] / lch_max
        
    for j in range(4):
        if sims[i][j] is None:
            results[i].append('-')
        else:
            results[i].append(f'{sims[i][j]:.6f}')

## Results

Now we proceed to print the similarities between words normalized as we explained before and in show a '-' in case of not available similarity.

In [5]:
print(tt.to_string(results, alignment='cccccc',
                   header=('Word 1', 'Word 2',
                           'Path Sim.', 'Leacock-Chodrow Sim.', 'Wu-Palmer Sim.', 'Lin Sim.')))

┌────────┬────────┬───────────┬──────────────────────┬────────────────┬──────────┐
│ Word 1 │ Word 2 │ Path Sim. │ Leacock-Chodrow Sim. │ Wu-Palmer Sim. │ Lin Sim. │
╞════════╪════════╪═══════════╪══════════════════════╪════════════════╪══════════╡
│  man   │  swim  │     -     │          -           │       -        │    -     │
├────────┼────────┼───────────┼──────────────────────┼────────────────┼──────────┤
│  man   │  girl  │ 0.250000  │       0.618897       │    0.631579    │ 0.713511 │
├────────┼────────┼───────────┼──────────────────────┼────────────────┼──────────┤
│  man   │  boy   │ 0.333333  │       0.697983       │    0.666667    │ 0.729472 │
├────────┼────────┼───────────┼──────────────────────┼────────────────┼──────────┤
│  man   │ woman  │ 0.333333  │       0.697983       │    0.666667    │ 0.787084 │
├────────┼────────┼───────────┼──────────────────────┼────────────────┼──────────┤
│  man   │  walk  │     -     │          -           │       -        │    -     │
├───

# Conclusions

Once we have normalized the similarities, we can visually compare them more easily and we can see that they share same similarity results. However, to be more precise, we are going to compute pearson correlation coefficients between the similarity methods to see if they give similar magnitude of scores for the same pairs of words. We are going to measure correlation against Path Similarity method as it is the simplest one.

In [6]:
M = np.array(results)

path = M[:,2]
l_ch = M[:,3]
wu = M[:,4]
lin = M[:,5]

p = [float(path[i]) for i in range(len(path)) if path[i] != '-' and l_ch[i]!='-']
l_ch = [float(l_ch[i]) for i in range(len(path)) if path[i] != '-' and l_ch[i]!='-']
print('Corr.\tPath Sim - Leacock-Chodrow Sim:\t', pearsonr(p,l_ch)[0])

p = [float(path[i]) for i in range(len(path)) if path[i] != '-' and wu[i]!='-']
wu = [float(wu[i]) for i in range(len(path)) if path[i] != '-' and wu[i]!='-']
print('Corr.\tPath Sim -       Wu-Palmer Sim:\t', pearsonr(p,wu)[0])

p = [float(path[i]) for i in range(len(path)) if path[i] != '-' and lin[i]!='-']
lin = [float(lin[i]) for i in range(len(path)) if path[i] != '-' and lin[i]!='-']
print('Corr.\tPath Sim -             Lin Sim:\t', pearsonr(p,lin)[0])

Corr.	Path Sim - Leacock-Chodrow Sim:	 0.932450431935749
Corr.	Path Sim -       Wu-Palmer Sim:	 0.7403675200774891
Corr.	Path Sim -             Lin Sim:	 0.8272809170757429


From the correlation results we can observe that coefficients are >= 0.75, which show that are highly correlated. Although there are a lot of missing values in the similarities outputs, the correlation obtained between the methods is quite high, which means that all of them perform pretty simmilar (at least for this concrete set of words from this exercise).

In the results table we can notice that there are some similarities that aren't equal comparing A with B, than B with A. Ideally NLTK's implementation of similarity would be totally symetric in behaviour, but because of its design, *path_similarity* across parts of speech is not commutative.

The method uses a dummy root, which allows to create a taxonomy to travel from one POS to another. A synset object has a method *\_\_need_root*, which may behave diferent depending of the synset. Hence, in practice, is not a symetric measure.